# Zero to Hero: Contrastive, Regularized & Joint Embeddings + Autoregressive EBMs

## Overview

Este notebook cubre tres pilares fundamentales del aprendizaje de representaciones moderno:

1. **Contrastive Methods vs Regularized Methods** — Cómo aprender representaciones sin labels
2. **Joint Embedding Architectures** — Arquitecturas que mapean inputs a espacios compartidos
3. **Autoregressive Generative Energy-Based Models** — Modelos generativos con energía

Cada sección va desde los fundamentos matemáticos hasta implementaciones from scratch.

### Prerequisites
```bash
pip install torch torchvision matplotlib numpy scikit-learn
```

## Part 0: Setup & Imports

---
# Part 0: Prerequisites — Lo que necesitas saber antes de empezar

Antes de meternos en las fórmulas y el código, repasemos los conceptos matemáticos y de ML que vas a necesitar. **No necesitas ser experto en nada de esto** — solo entender la intuición.

## 0.1 Álgebra Lineal Esencial

### Embeddings
Un **embedding** es simplemente un vector de números que representa algo (una imagen, un texto, un usuario):

$$z = f(x) \in \mathbb{R}^d$$

Por ejemplo, una imagen de gato podría ser:
$$z = [0.3, -1.2, 0.8, 2.1, ...] \in \mathbb{R}^{128}$$

**La idea clave:** cosas similares tienen embeddings cercanos en el espacio.

### Similitud coseno
¿Qué tan parecidos son dos embeddings? Medimos el ángulo entre ellos:

$$\text{sim}(a, b) = \frac{a \cdot b}{||a|| \cdot ||b||}$$

- **+1** = misma dirección (idénticos)
- **0** = perpendiculares (no relacionados)
- **-1** = direcciones opuestas (diferentes)

### Normalización L2
Antes de comparar, normalizamos vectores a longitud 1:

$$\hat{z} = \frac{z}{||z||_2}$$

Esto es equivalente a proyectar todos los puntos sobre una **esfera unitaria**. Después de normalizar, la similitud coseno es simplemente el producto punto.

### Matrices y producto matricial
Si tienes un batch de $N$ embeddings de dimensión $d$, los organizas en una matriz $[N \times d]$. El producto $Z \cdot Z^T$ te da una matriz $[N \times N]$ donde cada entrada $(i, j)$ es la similitud entre el embedding $i$ y el embedding $j$. **Esto es fundamental** para InfoNCE.

In [1]:
# Demo: Embeddings, normalización y similitud coseno
import torch
import torch.nn.functional as F

# Dos embeddings simulados (dim=5)
cat_emb = torch.tensor([0.3, -1.2, 0.8, 2.1, 0.5])
cat_emb_similar = torch.tensor([0.4, -1.0, 0.7, 2.0, 0.6])  # otro gato (similar)
dog_emb = torch.tensor([-0.5, 2.0, -1.0, 0.3, 1.8])  # un perro (diferente)

# Similitud coseno SIN normalizar
sim_cat_cat = F.cosine_similarity(cat_emb, cat_emb_similar, dim=0)
sim_cat_dog = F.cosine_similarity(cat_emb, dog_emb, dim=0)

print(f"Similitud coseno: gato vs gato-similar = {sim_cat_cat.item():.4f}")
print("  → Cerca de 1.0 significa que los embeddings apuntan en la misma dirección.")
print("  → Esto indica que el modelo reconoce que ambos son gatos.")
print()
print(f"Similitud coseno: gato vs perro = {sim_cat_dog.item():.4f}")
if sim_cat_dog > 0.5:
    print("  → Riesgo: el modelo confunde gatos con perros (similitud alta entre clases diferentes).")
else:
    print("  → Bueno: el modelo distingue bien entre clases (similitud baja entre clases diferentes).")
print()

# Normalización L2
cat_norm = F.normalize(cat_emb, dim=0)
print(f"Embedding original del gato:  [{cat_emb.tolist()}]")
print(f"Norma L2 original: {cat_emb.norm().item():.4f}")
print(f"Embedding normalizado:          [{cat_norm.tolist()}]")
print(f"Norma L2 normalizado: {cat_norm.norm().item():.4f}")
print("  → Después de normalizar, todos los embeddings viven en la superficie de una esfera.")
print("  → InfoNCE y las contrastive losses asumen embeddings normalizados para que la similitud sea coseno puro.")

Similitud coseno: gato vs gato-similar = 0.9958
  → Cerca de 1.0 significa que los embeddings apuntan en la misma dirección.
  → Esto indica que el modelo reconoce que ambos son gatos.

Similitud coseno: gato vs perro = -0.2377
  → Bueno: el modelo distingue bien entre clases (similitud baja entre clases diferentes).

Embedding original del gato:  [[0.30000001192092896, -1.2000000476837158, 0.800000011920929, 2.0999999046325684, 0.5]]
Norma L2 original: 2.6134
Embedding normalizado:          [[0.11479182541370392, -0.4591673016548157, 0.30611151456832886, 0.8035426735877991, 0.19131968915462494]]
Norma L2 normalizado: 1.0000
  → Después de normalizar, todos los embeddings viven en la superficie de una esfera.
  → InfoNCE y las contrastive losses asumen embeddings normalizados para que la similitud sea coseno puro.


## 0.2 Probabilidad y Función de Partición

### Distribución de Boltzmann
En los Energy-Based Models, la probabilidad se define como:

$$p(x) = \frac{e^{-E(x)}}{Z}$$

- $E(x)$ = energía del dato $x$ (baja = probable, alta = improbable)
- $Z = \int e^{-E(x)} dx$ = **función de partición** (constante de normalización)

### El problema de Z
Para un espacio de alta dimensión (como imágenes de 256×256), calcular $Z$ requiere integrar sobre $256^{256}$ posibilidades. **Es imposible**. Por eso los EBMs usan métodos aproximados (MCMC, Langevin sampling).

**Intuición:** No necesitas conocer $Z$ para comparar dos datos. Si $E(x_1) < E(x_2)$, entonces $p(x_1) > p(x_2)$ — sin importar el valor de $Z$.

### Softmax y temperatura
La temperatura $\tau$ controla qué tan "seguro" es el modelo:

$$p_i = \frac{e^{s_i / \tau}}{\sum_j e^{s_j / \tau}}$$

- **τ pequeño** (0.1): el modelo es muy confiado — solo el más similar tiene probabilidad alta
- **τ grande** (2.0): el modelo es suave — todas las opciones tienen probabilidad similar
- **τ = 1.0**: softmax estándar

In [2]:
# Demo: Efecto de la temperatura en la distribución
import torch
import torch.nn.functional as F

# Similitudes de un embedding con 5 candidatos (uno es el positivo)
similarities = torch.tensor([0.9, 0.1, 0.05, -0.1, -0.3])
print("Similitudes con 5 candidatos (el primero es el par positivo):")
print(f"  Positivo: {similarities[0].item():.2f}")
print(f"  Negativos: {similarities[1:].tolist()}")
print()

for tau in [0.1, 0.5, 1.0, 2.0]:
    probs = F.softmax(similarities / tau, dim=0)
    p_positive = probs[0].item()
    print(f"τ={tau:.1f}: P(positivo) = {p_positive:.4f}")
    if p_positive > 0.9:
        print("  → τ bajo: modelo muy seguro. El positivo domina la distribución.")
        print("  → La loss será pequeña porque el modelo ya identifica bien el positivo.")
    elif p_positive > 0.5:
        print("  → τ medio: positivo tiene mayoría pero hay incertidumbre.")
    else:
        print("  → τ alto: probabilidad se diluye. El modelo apenas distingue el positivo.")
        print("  → La loss será ALTA — el modelo necesita τ más bajo para enfocarse.")
    print()

Similitudes con 5 candidatos (el primero es el par positivo):
  Positivo: 0.90
  Negativos: [0.10000000149011612, 0.05000000074505806, -0.10000000149011612, -0.30000001192092896]

τ=0.1: P(positivo) = 0.9994
  → τ bajo: modelo muy seguro. El positivo domina la distribución.
  → La loss será pequeña porque el modelo ya identifica bien el positivo.

τ=0.5: P(positivo) = 0.6209
  → τ medio: positivo tiene mayoría pero hay incertidumbre.

τ=1.0: P(positivo) = 0.3928
  → τ alto: probabilidad se diluye. El modelo apenas distingue el positivo.
  → La loss será ALTA — el modelo necesita τ más bajo para enfocarse.

τ=2.0: P(positivo) = 0.2874
  → τ alto: probabilidad se diluye. El modelo apenas distingue el positivo.
  → La loss será ALTA — el modelo necesita τ más bajo para enfocarse.



## 0.3 Optimización y Backpropagation

### Gradient descent
Para minimizar una loss $L(\theta)$, actualizamos los parámetros:

$$\theta \leftarrow \theta - \eta \cdot \nabla_\theta L$$

donde $\eta$ es el learning rate.

### Conceptos clave que verás:

- **`loss.backward()`**: calcula los gradientes de todos los parámetros
- **`optimizer.step()`**: aplica los gradientes para actualizar los pesos
- **`optimizer.zero_grad()`**: limpia gradientes anteriores (se acumulan por defecto)
- **Learning rate**: muy grande → diverge (NaN). Muy pequeño → aprende muy lento.
- **Weight decay**: regularización L2 que penaliza pesos grandes → previene overfitting.

### EMA (Exponential Moving Average)
Usado en DINO e I-JEPA para el teacher encoder:

$$\theta_{teacher} \leftarrow m \cdot \theta_{teacher} + (1-m) \cdot \theta_{student}$$

donde $m \approx 0.996$. El teacher es una versión suavizada del student — evoluciona lentamente, dando targets estables.

In [3]:
# Demo: EMA — cómo el teacher evoluciona lentamente
import torch

student = torch.tensor([1.0, 2.0, 3.0])
teacher = student.clone()
momentum = 0.996

print("Evolución EMA del teacher (momentum = 0.996):")
print(f"  Inicial — student: {student.tolist()}, teacher: {teacher.tolist()}")

for step in range(1, 6):
    student_new = student + torch.randn(3) * 0.5
    teacher_new = momentum * teacher + (1 - momentum) * student_new
    s_vals = [round(v, 2) for v in student_new.tolist()]
    t_vals = [round(v, 2) for v in teacher_new.tolist()]
    print(f"  Paso {step} — student: {s_vals}, teacher: {t_vals}")
    student, teacher = student_new, teacher_new

print()
print("  → El teacher cambia MUY lentamente (99.6% de su valor anterior se mantiene).")
print("  → En DINO/I-JEPA esto es clave: el teacher da targets ESTABLES mientras el student aprende.")
print("  → Sin EMA, el teacher cambiaría tan rápido como el student → no habría señal consistente.")

Evolución EMA del teacher (momentum = 0.996):
  Inicial — student: [1.0, 2.0, 3.0], teacher: [1.0, 2.0, 3.0]
  Paso 1 — student: [0.58, 2.04, 2.96], teacher: [1.0, 2.0, 3.0]
  Paso 2 — student: [0.67, 2.74, 3.4], teacher: [1.0, 2.0, 3.0]
  Paso 3 — student: [0.8, 2.43, 3.12], teacher: [1.0, 2.0, 3.0]
  Paso 4 — student: [0.33, 2.73, 2.89], teacher: [0.99, 2.01, 3.0]
  Paso 5 — student: [-0.53, 2.36, 3.36], teacher: [0.99, 2.01, 3.0]

  → El teacher cambia MUY lentamente (99.6% de su valor anterior se mantiene).
  → En DINO/I-JEPA esto es clave: el teacher da targets ESTABLES mientras el student aprende.
  → Sin EMA, el teacher cambiaría tan rápido como el student → no habría señal consistente.


## 0.4 Conceptos de Machine Learning

### Self-Supervised Learning (SSL)
El modelo crea sus propios labels a partir de los datos sin etiquetar:

```
Imagen → [augmentación 1] → Vista A → Encoder → z_A
Imagen → [augmentación 2] → Vista B → Encoder → z_B
Objetivo: z_A y z_B deben ser similares (mismo dato, diferente vista)
```

**Augmentaciones comunes:** crop, color jitter, blur, rotación.

### El problema del colapso
Si solo minimizas la distancia entre $z_A$ y $z_B$, la solución trivial es $f(x) = c$ para todo $x$. **Todos los embeddings iguales → loss = 0 → representaciones inútiles.**

**Todos los métodos en este notebook resuelven el colapso de diferente forma:**

| Método | Solución al colapso |
|--------|--------------------|
| InfoNCE | Negativos explícitos: empuja embeddings diferentes lejos |
| Barlow Twins | Correlación cruzada → matriz identidad |
| VICReg | Penaliza varianza baja + correlación entre dimensiones |
| DINO/I-JEPA | Teacher EMA da targets estables |
| EBM | Energía con múltiples mínimos locales |

### Batch size y por qué importa
- **InfoNCE** necesita batch grande porque cada negativo viene del batch. Batch=32 → 62 negativos. Batch=4096 → 8190 negativos.
- **VICReg/Barlow** no necesitan negativos → funcionan con batch pequeño.
- **EBMs** son independientes del batch (pero MCMC es lento).

---

## 0.5 Setup — Imports y configuración

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_swiss_roll
from sklearn.manifold import TSNE
from itertools import combinations

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("  Excelente: los entrenamientos serán rápidos con GPU.")
else:
    print("  CPU mode: los entrenamientos serán lentos. Para demos está bien, pero en producción usa GPU.")
print()

torch.manual_seed(42)
np.random.seed(42)
print("Reproducibilidad: seed fijado a 42. Cada ejecución dará los mismos resultados.")

Using device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB
  Excelente: los entrenamientos serán rápidos con GPU.

Reproducibilidad: seed fijado a 42. Cada ejecución dará los mismos resultados.


---
# Part 1: Contrastive Methods vs Regularized Methods

## 1.1 ¿Qué problema resuelven?

Ambas familias buscan aprender **buenas representaciones** $f(x)$ de datos $x$ sin supervisión directa. La diferencia fundamental está en **cómo evitan el colapso** (todas las representaciones iguales).

### El problema del colapso
Si minimizas simplemente $||f(x_1) - f(x_2)||^2$ para augmentaciones del mismo dato, la solución trivial es $f(x) = c$ para todo $x$. Esto es inútil — necesitamos representaciones informativas.

---

## 1.2 Contrastive Methods

**Idea central:** Empujar muestras diferentes (negativos) lejos mientras acercas augmentaciones de la misma muestra (positivos).

### Contrastive Loss (clásica — Hadsell et al., 2006)

$$L = y \cdot d^2 + (1-y) \cdot \max(0, m - d)^2$$

donde $d = ||f(x_1) - f(x_2)||_2$, $y=1$ si son del mismo par, $m$ es margen.

### InfoNCE / NT-Xent (SimCLR)

La loss más influyente en self-supervised learning moderno:

$$L_i = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)}$$

- $z_i, z_j$: augmentaciones del mismo ejemplo (par positivo)
- $z_k$: todos los demás ejemplos en el batch (negativos)
- $\tau$: temperatura — controla qué tan "duro" es el contraste
- El denominador es una **softmax sobre similitudes** — el modelo debe identificar cuál es el positivo entre $2N-1$ candidatos

In [ ]:
z_i = torch.randn(8, 64)  # 8 muestras, dim 64
z_j = z_i + 0.1 * torch.randn(8, 64)  # augmentación cercana
loss = contrastive_loss_ntxent(z_i, z_j, temperature=0.5)
print(f"NT-Xent Loss: {loss.item():.4f}")
print(f"  → InfoNCE con batch=8 (16 embeddings total).")
print(f"  → Con 8 muestras, cada embedding tiene solo 14 negativos.",
       " Esto es POBRE — en SimCLR real se usan batch=4096 (8190 negativos).")
print(f"  → La loss baja indica que el modelo puede identificar el positivo entre los negativos.")
print()

# Verificar que la temperatura afecta:
print("Efecto de la temperatura τ:")
for tau in [0.1, 0.5, 1.0, 2.0]:
    l = contrastive_loss_ntxent(z_i, z_j, temperature=tau)
    print(f"  τ={tau:.1f} → loss={l.item():.4f}")
    if tau == 0.1 and l.item() > 3:
        print(f"    → τ muy bajo: la loss se dispara porque el modelo es demasiado confiado",
              "y los pequeños errores cuestan mucho.")
    elif tau == 2.0 and l.item() > 2.5:
        print(f"    → τ muy alto: la loss es alta porque el modelo no distingue bien",
              "el positivo de los negativos.")
print()
print("Regla práctica: τ ∈ [0.07, 0.5] funciona mejor para InfoNCE.",
      "τ más bajo = más contraste = gradiente más fuerte pero más inestable.")


### ¿Por qué funciona el contraste?

InfoNCE es equivalent a **maximizar un límite inferior de la información mutua** entre las representaciones de dos vistas del mismo dato:

$$I(Z_i; Z_j) \geq \log(N) - L_{\text{InfoNCE}}$$

donde $N$ es el número de negativos. Más negativos = límite más ajustado.

**Desventajas del enfoque contrastivo:**
- Requiere **muchos negativos** (batch sizes grandes, ej. 4096+ en SimCLR)
- La memoria crece cuadráticamente con el batch
- Negativos falsos: dos augmentaciones de clases diferentes que son semánticamente similares

---

## 1.3 Regularized Methods (Non-Contrastive)

**Idea central:** En lugar de usar negativos explícitos, usa **regularización** para prevenir el colapso. Solo trabajas con pares positivos.

### Barlow Twins (Zbontar et al., 2021)

Calcula la matriz de correlación cruzada entre las dimensiones de dos vistas y regulariza para que sea la identidad:

$$C_{ij} = \frac{\sum_b z^A_{b,i} \cdot z^B_{b,j}}{\sqrt{\sum_b (z^A_{b,i})^2} \sqrt{\sum_b (z^B_{b,j})^2}}$$

$$L = \underbrace{\sum_i (1 - C_{ii})^2}_{\text{invariance}} + \lambda \underbrace{\sum_i \sum_{j \neq i} C_{ij}^2}_{\text{redundancy reduction}}$$

- **Invariance:** Las diagonales deben ser 1 (las dos vistas dan el mismo embedding)
- **Redundancy:** Las off-diagonales deben ser 0 (cada dimensión captura info diferente)

### VICReg (Bardes et al., 2022)

Tres términos independientes:

$$L = \mu \cdot L_{\text{invariance}} + \nu \cdot L_{\text{variance}} + \gamma \cdot L_{\text{covariance}}$$

1. **Invariance:** MSE entre embeddings de augmentaciones (igual que SimCLR sin negativos)
2. **Variance:** Penaliza si la varianza de cualquier dimensión cae bajo un umbral (previene colapso dimensional)
3. **Covariance:** Penaliza correlaciones entre dimensiones (decorrelación, como Barlow Twins)

In [ ]:
z_i = torch.randn(256, 128)
z_j = z_i + 0.1 * torch.randn(256, 128)
loss_bt = barlow_twins_loss(z_i, z_j)
print(f"Barlow Twins Loss: {loss_bt.item():.4f}")
print(f"  → Batch=256, dim=128. La loss mide cuánto se aleja la correlación cruzada de la identidad.")
print(f"  → Loss baja (<10 para dim=128): las representaciones son informativas y no redundantes.")
print(f"  → Loss alta: las dimensiones están correlacionadas (redundancia) o no son invariantes.")
print()

# Caso de colapso: todos los embeddings iguales
z_collapsed = torch.ones(256, 128)
loss_collapsed = barlow_twins_loss(z_collapsed, z_collapsed)
print(f"Barlow Twins (colapso): {loss_collapsed.item():.4f}")
if loss_collapsed > 100:
    print("  → Loss ALTÍSIMA en colapso: Barlow Twins detecta que TODAS las dimensiones son idénticas.")
    print("  → La off-diagonal de la correlación cruzada = 1 (debería ser 0) → penalización masiva.")
    print("  → Barlow Twins NO puede colapsar — la regularización lo impide matemáticamente.")
else:
    print(f"  → Loss={loss_collapsed.item():.2f}: Barlow Twins penaliza el colapso pero no perfectamente.")
print()
print("Nota: Barlow Twins tiene O(D²) complejidad por la matriz de correlación cruzada.",
      "Con dim=128, eso es 16,384 entradas que regularizar. Por eso funciona con batch pequeño.")


In [ ]:
z_i = torch.randn(256, 128)
z_j = z_i + 0.1 * torch.randn(256, 128)
loss_vicreg = vicreg_loss(z_i, z_j)
print(f"VICReg Loss: {loss_vicreg.item():.4f}")
print(f"  → VICReg tiene 3 términos: invariance(25x) + variance(25x) + covariance(1x).")
print(f"  → Si la loss está dominada por invariance: las augmentaciones no son consistentes.")
print(f"  → Si está dominada por variance: las dimensiones colapsan a valores constantes.")
print(f"  → Si está dominada por covariance: las dimensiones capturan la misma información.")
print()

# Caso de colapso
z_collapsed = torch.ones(256, 128)
loss_collapsed_v = vicreg_loss(z_collapsed, z_collapsed)
print(f"VICReg (colapso): {loss_collapsed_v.item():.4f}")
if loss_collapsed_v > 40:
    print("  → Loss ALTÍSIMA: el término de variance detecta std=0 en todas las dimensiones.")
    print("  → ReLU(1 - 0) = 1 para cada dimensión → 128 × 25 = 3200 solo de variance.")
    print("  → VICReg NO puede colapsar — la varianza mínima está forzada por diseño.")
print()
print("Comparación rápida:")
print(f"  InfoNCE: necesita batch grande para tener suficientes negativos.")
print(f"  Barlow: no necesita negativos, pero O(D²) en memoria de correlación.")
print(f"  VICReg: más flexible que Barlow (3 coeficientes ajustables), funciona con batch muy pequeño.")


## 1.4 Comparación directa: Contrastive vs Regularized

| Aspecto | Contrastive (SimCLR) | Regularized (VICReg/Barlow) |
|---------|---------------------|----------------------------|
| **Mecanismo anti-colapso** | Negativos explícitos | Regularización en las dimensiones |
| **Tamaño de batch** | Necesita grande (512-8192) | Funciona con batch pequeño (64-256) |
| **Complejidad** | O(N²) por matriz de similitud | O(D²) por matriz de correlación |
| **Memoria** | Alta (todos los negativos) | Baja (solo pares positivos) |
| **Dimensionalidad** | Mejor con projector no-lineal | Funciona sin projector |
| **Rendimiento** | State-of-the-art en ImageNet | Competitivo, menos sensible a hiperparámetros |

### Insight clave

Ambas familias están **unificadas teóricamente**: el contraste maximiza información mutua implícitamente, mientras que la regularización maximiza la dimensionalidad efectiva del espacio latente. Son dos caminos al mismo destino.

In [ ]:
# Generar datos: Swiss Roll en 3D, proyectar a 2D
n_samples = 2000
X, color = make_swiss_roll(n_samples, noise=0.3)
X = X[:, [0, 2]].astype(np.float32)  # Proyectar a 2D para visualizar

# Normalizar
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
X_tensor = torch.tensor(X, device=device)

# Dataset simple: cada punto es su propia augmentación con ruido
def augment(x, noise_level=0.15):
    return x + noise_level * torch.randn_like(x)

# Encoder simple: MLP
class SimpleEncoder(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, output_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)

print(f"Datos Swiss Roll 2D generados: {X.shape[0]} muestras, {X.shape[1]} dimensiones")
print(f"  Rango X: min={X.min(axis=0).values}, max={X.max(axis=0).values}")
print("  Los datos están normalizados (media≈0, std≈1) para que el encoder no tenga que aprender escalas.")
print("  El Swiss Roll es un manifold no-lineal — si el embedding preserva la estructura,",
      "significa que aprendió la geometría intrínseca, no solo proyección lineal.")

# Visualizar datos originales
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(X[:, 0], X[:, 1], c=color, cmap="viridis", s=10, alpha=0.7)
axes[0].set_title("Datos originales (Swiss Roll 2D)")
axes[0].set_xlabel("x₁")
axes[0].set_ylabel("x₂")
axes[0].axis("equal")
plt.tight_layout()
plt.show()


In [ ]:
def train_encoder(encoder, loss_fn, optimizer, n_epochs=200, batch_size=128, noise_level=0.15):
    """Entrenar encoder con la loss dada."""
    losses = []
    n = X_tensor.shape[0]

    for epoch in range(n_epochs):
        idx = torch.randperm(n, device=device)[:batch_size]
        x_batch = X_tensor[idx]

        z_i = encoder(augment(x_batch, noise_level))
        z_j = encoder(augment(x_batch, noise_level))

        loss = loss_fn(z_i, z_j)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if (epoch + 1) % 50 == 0:
            if epoch == 49:
                print(f"  Epoch {epoch+1}/{n_epochs} — Loss: {loss.item():.4f}", end="")
                if loss.item() > 5:
                    print(" → Loss alta al inicio: el encoder aún no ha aprendido a distinguir vistas.")
                else:
                    print(" → Loss baja temprano: el encoder converge rápido o los datos son fáciles.")
            elif epoch == n_epochs - 1:
                print(f"  Epoch {epoch+1}/{n_epochs} — Loss: {loss.item():.4f}", end="")
                if loss.item() < 1:
                    print(" → Loss final baja: el modelo aprendió buenas representaciones.")
                else:
                    print(" → Loss aún alta: podría necesitar más epochs, learning rate más alto, o mejor arquitectura.")
                print()
            else:
                print(f"  Epoch {epoch+1}/{n_epochs} — Loss: {loss.item():.4f}")

    return losses

# Entrenar con InfoNCE (contrastive)
print("\n=== Entrenando con InfoNCE (Contrastive) ===")
print("  InfoNCE empuja augmentaciones del mismo dato cerca y aleja las de datos diferentes.")
print("  Con batch=256, cada embedding tiene 510 negativos — señal fuerte pero limitada vs batch=4096.")
encoder_contrastive = SimpleEncoder(input_dim=2, hidden_dim=64, output_dim=16).to(device)
opt_c = torch.optim.Adam(encoder_contrastive.parameters(), lr=1e-3)
losses_c = train_encoder(encoder_contrastive, contrastive_loss_ntxent, opt_c, n_epochs=300, batch_size=256)
print(f"  Loss InfoNCE final: {losses_c[-1]:.4f}")
if losses_c[-1] < 2.0:
    print("  → Buen resultado: el encoder contrastivo aprendió a separar clases en el manifold.")
elif losses_c[-1] < 3.5:
    print("  → Resultado moderado: InfoNCE con batch=256 no tiene suficientes negativos para este dataset.")
else:
    print("  → Loss alta: InfoNCE necesita batch más grande o learning rate diferente para este problema.")
print()

# Entrenar con Barlow Twins (regularized)
print("\n=== Entrenando con Barlow Twins (Regularized) ===")
print("  Barlow Twins no usa negativos — solo fuerza correlación cruzada → identidad.")
print("  Debería funcionar bien incluso con batch=256 porque no depende del número de negativos.")
encoder_barlow = SimpleEncoder(input_dim=2, hidden_dim=64, output_dim=16).to(device)
opt_b = torch.optim.Adam(encoder_barlow.parameters(), lr=1e-3)
losses_b = train_encoder(encoder_barlow, barlow_twins_loss, opt_b, n_epochs=300, batch_size=256)
print(f"  Loss Barlow final: {losses_b[-1]:.4f}")
if losses_b[-1] < 10:
    print("  → Buen resultado: la correlación cruzada está cerca de la identidad.")
else:
    print("  → Loss alta: las dimensiones aún están correlacionadas o no son invariantes.")

# Plot de losses
plt.figure(figsize=(10, 4))
plt.plot(losses_c, label="InfoNCE (Contrastive)", alpha=0.8)
plt.plot(losses_b, label="Barlow Twins (Regularized)", alpha=0.8)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss: Contrastive vs Regularized")
plt.legend()
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.show()

# Interpretación del gráfico
print("\nInterpretando el gráfico:")
print("  → Si InfoNCE baja más rápido: los negativos dan señal más fuerte para este dataset.")
print("  → Si Barlow baja más estable: la regularización es más robusta sin depender de negativos.")
print("  → En la práctica, Barlow Twins suele ser más estable con batch pequeño.")


In [ ]:
encoder_contrastive.eval()
encoder_barlow.eval()

with torch.no_grad():
    z_contrastive = encoder_contrastive(X_tensor).cpu().numpy()
    z_barlow = encoder_barlow(X_tensor).cpu().numpy()

# t-SNE a 2D para visualizar
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
z_contrastive_2d = tsne.fit_transform(z_contrastive)
z_barlow_2d = tsne.fit_transform(z_barlow)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X[:, 0], X[:, 1], c=color, cmap="viridis", s=10, alpha=0.7)
axes[0].set_title("Datos originales")
axes[0].axis("equal")

axes[1].scatter(z_contrastive_2d[:, 0], z_contrastive_2d[:, 1], c=color, cmap="viridis", s=10, alpha=0.7)
axes[1].set_title("InfoNCE embeddings (t-SNE)")
axes[1].axis("equal")

axes[2].scatter(z_barlow_2d[:, 0], z_barlow_2d[:, 1], c=color, cmap="viridis", s=10, alpha=0.7)
axes[2].set_title("Barlow Twins embeddings (t-SNE)")
axes[2].axis("equal")

plt.tight_layout()
plt.show()

print("¿Qué buscar en estas visualizaciones?")
print("  1. Estructura del manifold: si el Swiss Roll se desenrolla correctamente, el embedding preserva la geometría.")
print("  2. Separación de colores: regiones cercanas en el original deben estar cercanas en el embedding.")
print("  3. Colapso: si todos los puntos se juntan en un solo cluster, el modelo colapsó (no pasó aquí gracias a las losses).")
print("  → InfoNCE tiende a separar más las clases diferentes (push apart negativos).")
print("  → Barlow Twins tiende a preservar la estructura local sin forzar separación extrema.")


---
# Part 2: Joint Embedding Architectures

## 2.1 ¿Qué es un Joint Embedding?

Un **joint embedding architecture** (JEA) mapea dos (o más) inputs a un **espacio compartido** donde se puede medir su relación. La idea es que inputs semánticamente relacionados estén cerca en el espacio embedding, sin importar su modalidad o augmentación.

### Formulación general

```
Input A ──→ Encoder_A ──→ Projector_A ─→ z_A ──┐
                                               ├──→ Loss(z_A, z_B)
Input B ──→ Encoder_B ──→ Projector_B ─→ z_B ──┘
```

El "joint" viene de que ambos se entrenan para que sus embeddings sean **comparables** en el mismo espacio.

## 2.2 Arquitecturas principales

### Siamese Networks (clásico)
Dos ramas con **pesos compartidos**. Mismo encoder para ambos inputs.

### Two-Tower / Dual Encoder
Dos encoders **independientes**. Útil cuando los inputs son de diferente naturaleza (texto vs imagen).

### Asymmetric Siamese (DINO, DINOv2)
Dos encoders con pesos compartidos pero **augmentaciones asimétricas**: un "teacher" (EMA del student) y un "student". El teacher genera targets, el student aprende a predecirlos.

### I-JEPA (Joint-Embedding Predictive Architecture)
Yann LeCun's propuesta: en lugar de predecir píxeles (generativo) o contrastar con negativos, **predice embeddings de regiones ocultas** en el espacio abstracto.

```
Imagen completa ──→ Context Encoder ─→ z_context ──┐
                                                  ├──→ Predictor ─→ z_pred ≈ z_target
Imagen con mask ──→ Target Encoder  ─→ z_target ──┘
```

La pérdida es MSE entre $z_{pred}$ y $z_{target}$, con regularización de varianza para evitar colapso.

In [ ]:
# ============================================================
# IMPLEMENTACIÓN: I-JEPA simplificado
# ============================================================

class ContextEncoder(nn.Module):
    """Encoder que ve la imagen completa (o contexto visible)."""
    def __init__(self, input_dim=16, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, embed_dim),
        )

    def forward(self, x):
        return self.net(x)


class TargetEncoder(nn.Module):
    """Encoder que ve solo las regiones target (masked)."""
    def __init__(self, input_dim=16, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, embed_dim),
        )

    def forward(self, x):
        return self.net(x)


class Predictor(nn.Module):
    """Mapea z_context → z_pred para comparar con z_target."""
    def __init__(self, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Linear(64, embed_dim),
        )

    def forward(self, z):
        return self.net(z)


class IJEPA(nn.Module):
    """
    I-JEPA simplificado.

    Filosofía:
    1. El context encoder ve la parte visible de la imagen
    2. El target encoder ve las regiones masked
    3. El predictor mapea contexto → embedding del target
    4. Loss: MSE(z_pred, z_target) con regularización de varianza

    No usa negativos, no usa augmentaciones agresivas.
    El modelo aprende a 'imaginar' lo que falta en el espacio abstracto.
    """
    def __init__(self, input_dim=16, embed_dim=32):
        super().__init__()
        self.context_encoder = ContextEncoder(input_dim, embed_dim)
        self.target_encoder = TargetEncoder(input_dim, embed_dim)
        self.predictor = Predictor(embed_dim)

        # Target encoder es EMA del context (como DINO)
        self._init_target_encoder()

    def _init_target_encoder(self):
        # Copiar pesos iniciales
        for p_ctx, p_tgt in zip(
            self.context_encoder.parameters(),
            self.target_encoder.parameters()
        ):
            p_tgt.data.copy_(p_ctx.data)
            p_tgt.requires_grad = False

    @torch.no_grad()
    def update_target_encoder(self, momentum=0.996):
        """EMA update: θ_target = m·θ_target + (1-m)·θ_context"""
        for p_ctx, p_tgt in zip(
            self.context_encoder.parameters(),
            self.target_encoder.parameters()
        ):
            p_tgt.data = momentum * p_tgt.data + (1 - momentum) * p_ctx.data

    def forward(self, x_context, x_target):
        z_context = self.context_encoder(x_context)
        z_pred = self.predictor(z_context)

        with torch.no_grad():
            z_target = self.target_encoder(x_target)

        return z_pred, z_target


def ijepa_loss(z_pred, z_target):
    """
    Loss de I-JEPA: MSE + regularización de varianza.
    Similar a VICReg pero aplicado a predicción en espacio latente.
    """
    # Invariance: MSE entre predicción y target
    mse = F.mse_loss(z_pred, z_target)

    # Varianza: asegurar que las dimensiones no colapsen
    std_pred = torch.sqrt(z_pred.var(dim=0) + 1e-4)
    std_target = torch.sqrt(z_target.var(dim=0) + 1e-4)
    var_loss = F.relu(1 - std_pred).mean() + F.relu(1 - std_target).mean()

    return mse + 0.5 * var_loss


# Demo
model = IJEPA(input_dim=16, embed_dim=32).to(device)
x_ctx = torch.randn(32, 16, device=device)  # contexto visible
x_tgt = torch.randn(32, 16, device=device)  # región masked
z_pred, z_target = model(x_ctx, x_tgt)
loss = ijepa_loss(z_pred, z_target)
print(f"I-JEPA Loss: {loss.item():.4f}")
if loss.item() < 1.0:
    print("  → Loss baja: el predictor ya puede aproximar el target en espacio latente.")
    print("  → Con datos aleatorios esto es esperado — las dos vistas son similares por ruido.")
elif loss.item() < 2.0:
    print("  → Loss moderada: el predictor aún no converge bien con pesos aleatorios.")
else:
    print("  → Loss alta: el predictor necesita entrenamiento para aprender a predecir.")
print(f"z_pred shape: {z_pred.shape}, z_target shape: {z_target.shape}")
print("  → Ambas tienen la misma dimensión (embed_dim=32) — son comparables directamente.")
print("  → El contexto (primera mitad de la imagen) se mapea al espacio del target (segunda mitad).")


In [ ]:
# ============================================================
# Entrenar I-JEPA en datos sintéticos
# ============================================================

# Datos: vectores de alta dimensión con estructura
n_samples = 5000
input_dim = 32
embed_dim = 16

# Crear datos con estructura latente: x = f(latent) + noise
latent = torch.randn(n_samples, 8, device=device)
W_true = torch.randn(8, input_dim, device=device)
X_data = latent @ W_true + 0.1 * torch.randn(n_samples, input_dim, device=device)

# Simular masking: contexto = algunas dimensiones, target = otras
mask_context = torch.ones(input_dim, device=device)
mask_context[input_dim // 2:] = 0  # primera mitad visible
mask_target = 1 - mask_context      # segunda mitad oculta

def mask_data(x, mask):
    return x * mask

# Entrenar
model = IJEPA(input_dim=input_dim, embed_dim=embed_dim).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

losses_ijepa = []
batch_size = 128
n_epochs = 500

for epoch in range(n_epochs):
    idx = torch.randperm(n_samples, device=device)[:batch_size]
    x_batch = X_data[idx]

    x_ctx = mask_data(x_batch, mask_context)
    x_tgt = mask_data(x_batch, mask_target)

    z_pred, z_target = model(x_ctx, x_tgt)
    loss = ijepa_loss(z_pred, z_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.update_target_encoder(momentum=0.996)
    losses_ijepa.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch+1}/{n_epochs} — Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses_ijepa, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('I-JEPA Loss')
plt.title('I-JEPA Training: predicting masked regions in latent space')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# IMPLEMENTACIÓN: Two-Tower Model (Cross-Modal Joint Embedding)
# ============================================================

class TwoTowerModel(nn.Module):
    """
    Two-Tower para embeddings conjuntos de dos modalidades.
    Ejemplo: texto ↔ imagen, o audio ↔ video.

    Cada tower mapea su input a un espacio compartido.
    Usamos InfoNCE para alinear los espacios.
    """
    def __init__(self, dim_a=64, dim_b=64, embed_dim=32):
        super().__init__()
        self.tower_a = nn.Sequential(
            nn.Linear(dim_a, 128),
            nn.ReLU(),
            nn.Linear(128, embed_dim),
        )
        self.tower_b = nn.Sequential(
            nn.Linear(dim_b, 128),
            nn.ReLU(),
            nn.Linear(128, embed_dim),
        )
        self.temperature = nn.Parameter(torch.tensor(0.07))

    def forward(self, x_a, x_b):
        z_a = F.normalize(self.tower_a(x_a), dim=-1)
        z_b = F.normalize(self.tower_b(x_b), dim=-1)
        return z_a, z_b, self.temperature


def two_tower_loss(z_a, z_b, temperature):
    """InfoNCE bidireccional para two-tower."""
    # Similarity matrix
    sim = (z_a @ z_b.T) / temperature

    # Labels: diagonal son los pares correctos
    labels = torch.arange(z_a.shape[0], device=z_a.device)

    # Loss a→b y b→a
    loss_a = F.cross_entropy(sim, labels)
    loss_b = F.cross_entropy(sim.T, labels)

    return (loss_a + loss_b) / 2


# Demo: simular pares texto-imagen
n_pairs = 500
dim_text = 128  # Simulación de embeddings de texto
dim_image = 256  # Simulación de embeddings de imagen

# Datos: pares correlacionados en un espacio latente
latent_shared = torch.randn(n_pairs, 16, device=device)
W_text = torch.randn(16, dim_text, device=device)
W_image = torch.randn(16, dim_image, device=device)
text_data = latent_shared @ W_text + 0.5 * torch.randn(n_pairs, dim_text, device=device)
image_data = latent_shared @ W_image + 0.5 * torch.randn(n_pairs, dim_image, device=device)

# Entrenar
model_tt = TwoTowerModel(dim_text, dim_image, embed_dim=32).to(device)
opt_tt = torch.optim.AdamW(model_tt.parameters(), lr=1e-3)

losses_tt = []
for epoch in range(300):
    idx = torch.randperm(n_pairs, device=device)[:64]
    z_a, z_b, temp = model_tt(text_data[idx], image_data[idx])
    loss = two_tower_loss(z_a, z_b, temp)
    opt_tt.zero_grad()
    loss.backward()
    opt_tt.step()
    losses_tt.append(loss.item())
    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch+1}/300 — Loss: {loss.item():.4f}, temp: {temp.item():.3f}")
        if loss.item() < 1.0:
            print("    → Loss baja: las dos towers están alineando sus espacios de embedding correctamente.")
        elif loss.item() < 3.0:
            print("    → Loss moderada: las towers están aprendiendo pero aún hay confusión entre pares.")
        else:
            print("    → Loss alta: las towers no han encontrado la correspondencia cross-modal aún.")
        if temp.item() < 0.05:
            print("    → Temperatura muy baja: el modelo se volvió demasiado confiado, riesgo de inestabilidad.")
        elif temp.item() > 0.5:
            print("    → Temperatura alta: el modelo aún no distingue bien los pares correctos.")

# Evaluar: calcular accuracy de retrieval
model_tt.eval()
with torch.no_grad():
    z_all_a, z_all_b, _ = model_tt(text_data, image_data)
    sim_matrix = z_all_a @ z_all_b.T
    predictions = sim_matrix.argmax(dim=1)
    labels = torch.arange(n_pairs, device=device)
    accuracy = (predictions == labels).float().mean()
    accuracy_pct = accuracy.item() * 100
    print(f"\nRetrieval accuracy: {accuracy_pct:.1f}%")
    if accuracy_pct > 90:
        print("  → Excelente: casi todos los textos recuperan la imagen correcta y viceversa.")
        print("  → Los espacios de embedding están bien alineados entre las dos modalidades.")
    elif accuracy_pct > 50:
        print("  → Moderado: el modelo encuentra el par correcto más que al azar (1/n_pairs).")
        print("  → Podría mejorar con más epochs, más datos, o un learning rate diferente.")
    else:
        print("  → Bajo: el modelo no logró alinear las dos modalidades efectivamente.")
        print("  → Posible causa: los datos no tienen suficiente señal compartida, o el modelo es muy pequeño.")

plt.figure(figsize=(8, 4))
plt.plot(losses_tt, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Two-Tower InfoNCE Loss')
plt.title('Two-Tower Joint Embedding Training')
plt.grid(True, alpha=0.3)
plt.show()

---
# Part 3: Autoregressive Generative Energy-Based Models

## 3.1 Energy-Based Models (EBMs)

### Idea fundamental

Un EBM define una distribución de probabilidad a través de una **función de energía** $E_\theta(x)$:

$$p_\theta(x) = \frac{e^{-E_\theta(x)}}{Z_\theta}$$

donde $Z_\theta = \int e^{-E_\theta(x)} dx$ es la **función de partición** (normalización).

- **Baja energía** → alta probabilidad (datos "buenos")
- **Alta energía** → baja probabilidad (datos "malos")

### El problema de $Z_\theta$

La integral de partición es **intratable** para espacios de alta dimensión. Esto hace que:
- Calcular $p_\theta(x)$ exacto es imposible
- El gradiente de la log-likelihood requiere muestreo (MCMC)

### Gradiente de la log-likelihood

$$\nabla_\theta \log p_\theta(x) = -\mathbb{E}_{p_\theta(x)}[\nabla_\theta E_\theta(x)] + \mathbb{E}_{x \sim \text{data}}[\nabla_\theta E_\theta(x)]$$

- **Fase positiva:** bajar energía de datos reales
- **Fase negativa:** subir energía de muestras del modelo (MCMC)

## 3.2 Modelos Autoregresivos

Un modelo autoregresivo factoriza la distribución:

$$p(x) = \prod_{t=1}^{T} p(x_t | x_{<t})$$

Cada paso predice el siguiente elemento condicionado a los anteriores. **No tiene problema de partición** porque normaliza localmente en cada paso.

**Ventajas:**
- Entrenamiento exacto por maximum likelihood
- Muestreo directo (forward pass)

**Desventajas:**
- Secuencial (lento para muestreo)
- Expositional bias: errores se acumulan

## 3.3 Autoregressive EBMs: Lo mejor de ambos mundos

La idea es **combinar** la flexibilidad de los EBMs con la tractabilidad de los modelos autoregresivos:

### Approach 1: AR como proposal para EBM
Usar un modelo AR $p_{AR}(x)$ como distribución propuesta, y refinarlo con un EBM:

$$p(x) \propto p_{AR}(x) \cdot e^{-E_{EBM}(x)}$$

El AR genera muestras rápidas, el EBM les asigna energía y filtra las malas.

### Approach 2: Energy-based autoregressive decoding
En cada paso del decoder, en lugar de solo maximizar $p(x_t | x_{<t})$, se minimiza una energía que captura restricciones globales:

$$p(x_t | x_{<t}) \propto \exp(-E(x_t, x_{<t}))$$

### Approach 3: Generative Flow + EBM
Un AR model produce una secuencia latente $z$, y un EBM refina $z$ antes de decodificar:

$$z \sim p_{AR}(z) \rightarrow z' = \text{Langevin}(z, E_{EBM}) \rightarrow x = \text{decoder}(z')$$

In [ ]:
# ============================================================
# IMPLEMENTACIÓN: EBM básico con Langevin Sampling
# ============================================================

class SimpleEBM(nn.Module):
    """
    Energy-Based Model simple.
    E_θ(x) = MLP(x) → escalar de energía

    Entrenamiento: contrastive divergence aproximado.
    """
    def __init__(self, input_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

    def energy(self, x):
        """E_θ(x) → [batch, 1]"""
        return self.net(x).squeeze(-1)

    def sample_langevin(self, x_init, n_steps=20, step_size=0.01, noise_std=0.01):
        """
        Langevin MCMC: x_{t+1} = x_t - ε/2 · ∇E(x_t) + √ε · η

        Esto muestrea de p(x) ∝ exp(-E(x)) sin conocer Z.
        """
        x = x_init.clone().requires_grad_(True)

        for _ in range(n_steps):
            energy = self.energy(x).sum()
            grad = torch.autograd.grad(energy, x)[0]

            # Update: gradient descent en energía + noise
            x = x - (step_size / 2) * grad + noise_std * torch.randn_like(x)
            x = x.detach().requires_grad_(True)

        return x


def contrastive_divergence_loss(ebm, x_real, x_fake):
    """
    CD-k loss:
    L = E_θ(x_real) - E_θ(x_fake)

    Minimizar esto baja la energía de datos reales y sube la de fakes.
    """
    return ebm.energy(x_real).mean() - ebm.energy(x_fake).mean()


# Demo: EBM en datos 2D de two moons
X_moons, _ = make_moons(n_samples=500, noise=0.1)
X_moons = torch.tensor(X_moons, dtype=torch.float32, device=device)

ebm = SimpleEBM(input_dim=2).to(device)
optimizer_ebm = torch.optim.Adam(ebm.parameters(), lr=1e-3)

losses_ebm = []
for epoch in range(500):
    # Datos reales
    idx = torch.randperm(X_moons.shape[0])[:64]
    x_real = X_moons[idx]

    # Muestrear con Langevin desde ruido
    x_init = torch.randn(64, 2, device=device) * 2
    x_fake = ebm.sample_langevin(x_init, n_steps=10, step_size=0.1, noise_std=0.05)

    loss = contrastive_divergence_loss(ebm, x_real, x_fake)
    optimizer_ebm.zero_grad()
    loss.backward()
    optimizer_ebm.step()

    losses_ebm.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch+1}/500 — CD Loss: {loss.item():.4f}")
        if loss.item() < -1:
            print("    → Loss negativa: E(x_real) << E(x_fake). El EBM distingue bien datos reales de fake.")
            print("    → El modelo aprendió a asignar baja energía a la distribución real.")
        elif loss.item() < 0:
            print("    → Loss ligeramente negativa: progreso, pero el EBM aún confunde algunos reales con fakes.")
        else:
            print("    → Loss positiva: E(x_real) > E(x_fake). El EBM aún no aprendió la distribución.")
            print("    → Puede necesitar más pasos de Langevin (mejor muestreo) o learning rate diferente.")

plt.figure(figsize=(8, 4))
plt.plot(losses_ebm, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Contrastive Divergence Loss')
plt.title('EBM Training with Langevin Sampling')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Visualizar la landscape de energía aprendida
# ============================================================

ebm.eval()

# Grid para visualizar energía
x_range = np.linspace(-3, 3, 100)
y_range = np.linspace(-3, 3, 100)
xx, yy = np.meshgrid(x_range, y_range)
grid = torch.tensor(np.stack([xx.ravel(), yy.ravel()], axis=1), dtype=torch.float32, device=device)

with torch.no_grad():
    energies = ebm.energy(grid).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Datos reales
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c='blue', s=10, alpha=0.5)
axes[0].set_title('Datos originales (Two Moons)')
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-3, 3)

# Energy landscape
energy_grid = energies.reshape(100, 100)
# Invertir: baja energía = alta probabilidad = colores cálidos
cf = axes[1].contourf(xx, yy, -energy_grid, levels=30, cmap='hot')
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c='blue', s=5, alpha=0.5)
axes[1].set_title('Energy Landscape (inverted: warm = low E)')
axes[1].set_xlim(-3, 3)
axes[1].set_ylim(-3, 3)
plt.colorbar(cf, ax=axes[1], label='-Energy (higher = more likely)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# IMPLEMENTACIÓN: Autoregressive EBM (AR + Energy Refinement)
# ============================================================

class AutoregressiveModel(nn.Module):
    """
    Modelo autoregresivo simple para datos secuenciales.
    x_t = f(x_{<t}) para cada paso t.
    """
    def __init__(self, seq_len=10, dim=8, hidden=32):
        super().__init__()
        self.seq_len = seq_len
        self.dim = dim
        self.embedding = nn.Embedding(100, dim)  # Vocabulario de 100

        # Transformer decoder simple (sin pos encoding para claridad)
        self.transformer = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(
                d_model=dim, nhead=4, dim_feedforward=hidden, batch_first=True
            ),
            num_layers=2
        )

        self.output_layer = nn.Linear(dim, 100)  # Logits sobre vocab

    def forward(self, x):
        """
        x: [batch, seq_len] con token IDs
        Returns: logits [batch, seq_len, vocab]
        """
        embedded = self.embedding(x)  # [batch, seq_len, dim]

        # Máscara causal: no ver el futuro
        causal_mask = torch.triu(
            torch.full((x.shape[1], x.shape[1]), float('-inf'), device=x.device),
            diagonal=1
        )

        output = self.transformer(embedded, memory=None, tgt_mask=causal_mask)
        return self.output_layer(output)


class AutoregressiveEBM(nn.Module):
    """
    Combina AR model con EBM para refinamiento.

    Flujo:
    1. AR model genera secuencia x_AR
    2. EBM asigna energía E(x_AR)
    3. Si E es alta, refinar con Langevin en espacio de embeddings
    4. Decodificar secuencia refinada

    Durante entrenamiento:
    - AR se entrena con NLL estándar
    - EBM se entrena con contrastive divergence
    - Gradientes del EBM pueden fluir de vuelta al AR (opcional)
    """
    def __init__(self, seq_len=10, dim=8, hidden=32, vocab=100):
        super().__init__()
        self.ar_model = AutoregressiveModel(seq_len, dim, hidden)
        self.ebm_energy = SimpleEBM(input_dim=dim * seq_len)
        self.vocab = vocab
        self.seq_len = seq_len
        self.dim = dim

    def ar_loss(self, x):
        """NLL del modelo autoregresivo."""
        logits = self.ar_model(x[:, :-1])  # [batch, seq_len-1, vocab]
        targets = x[:, 1:]  # [batch, seq_len-1]
        return F.cross_entropy(logits.reshape(-1, self.vocab), targets.reshape(-1))

    def sample_ar(self, batch_size, temp=1.0):
        """Muestreo autoregresivo greedy/temperature."""
        tokens = torch.zeros(batch_size, 1, dtype=torch.long, device=device)

        for _ in range(self.seq_len - 1):
            logits = self.ar_model(tokens)[:, -1, :] / temp
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            tokens = torch.cat([tokens, next_token], dim=1)

        return tokens

    def energy_of_sequence(self, tokens):
        """Energía EBM de una secuencia."""
        embedded = self.ar_model.embedding(tokens)  # [batch, seq_len, dim]
        flat = embedded.reshape(tokens.shape[0], -1)  # [batch, seq_len*dim]
        return self.ebm_energy.energy(flat)


# Demo
ar_ebm = AutoregressiveEBM(seq_len=8, dim=16, hidden=64, vocab=50).to(device)

# Generar datos sintéticos secuenciales
n_sequences = 1000
seq_len = 8
vocab = 50
synthetic_seqs = torch.randint(0, vocab, (n_sequences, seq_len), device=device)

# Entrenar AR component
print("\nEntrenando AR component (NLL estándar)...")
print("  El modelo AR aprende a predecir el siguiente token dado los anteriores.")
print("  NLL bajo = el modelo asigna alta probabilidad a la secuencia correcta.")
print("  Con vocab=50 y seq_len=8, el NLL mínimo teórico es log(50) ≈ 3.91 (distribución uniforme).")
opt_ar = torch.optim.Adam(ar_ebm.ar_model.parameters(), lr=1e-3)
ar_losses = []

for epoch in range(200):
    idx = torch.randperm(n_sequences)[:64]
    loss = ar_ebm.ar_loss(synthetic_seqs[idx])
    opt_ar.zero_grad()
    loss.backward()
    opt_ar.step()
    ar_losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        nll = loss.item()
        print(f"  Epoch {epoch+1}/200 — AR NLL: {nll:.4f}")
        if nll < 2.0:
            print(f"    → NLL muy bajo: el modelo AR predice los tokens con alta confianza.")
        elif nll < 3.5:
            print(f"    → NLL moderado: el modelo aprendió patrones pero aún tiene incertidumbre.")
        else:
            print(f"    → NLL alto (cerca de log(50)≈3.91): el modelo apenas está por encima de adivinar al azar.")

# Entrenar EBM component
print("\nEntrenando EBM component (Contrastive Divergence)...")
print("  El EBM aprende a distinguir embeddings de datos reales vs generados por el AR.")
print("  CD negativo = el EBM asigna menos energía a reales que a fakes (lo que queremos).")
opt_ebm = torch.optim.Adam(ar_ebm.ebm_energy.parameters(), lr=1e-3)
ebm_losses_ar = []

for epoch in range(200):
    idx = torch.randperm(n_sequences)[:32]
    x_real = ar_ebm.ar_model.embedding(synthetic_seqs[idx])
    x_real_flat = x_real.reshape(32, -1)

    # Fake: muestrear del AR
    with torch.no_grad():
        x_fake_tokens = ar_ebm.sample_ar(32)
    x_fake = ar_ebm.ar_model.embedding(x_fake_tokens)
    x_fake_flat = x_fake.reshape(32, -1)

    loss = contrastive_divergence_loss(ar_ebm.ebm_energy, x_real_flat, x_fake_flat)
    opt_ebm.zero_grad()
    loss.backward()
    opt_ebm.step()
    ebm_losses_ar.append(loss.item())
    if (epoch + 1) % 50 == 0:
        cd = loss.item()
        print(f"  Epoch {epoch+1}/200 — EBM CD: {cd:.4f}")
        if cd < -0.5:
            print("    → CD negativo fuerte: el EBM distingue bien entre reales y generados por el AR.")
            print("    → El AR necesita mejorar para 'engañar' al EBM.")
        elif cd < 0:
            print("    → CD ligeramente negativo: el EBM está aprendiendo pero aún hay solapamiento.")
        else:
            print("    → CD positivo: el EBM aún asigna más energía a reales que a fakes. Necesita más entrenamiento.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ar_losses, alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AR NLL')
axes[0].set_title('Autoregressive Model Training')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ebm_losses_ar, alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('EBM CD Loss')
axes[1].set_title('EBM Refinement Training')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Demo: Generar muestras y refinarlas con el EBM
# ============================================================

ar_ebm.eval()

# 1. Generar muestras del AR
with torch.no_grad():
    ar_samples = ar_ebm.sample_ar(5, temp=0.8)
    ar_energies = ar_ebm.energy_of_sequence(ar_samples)

print("\nMuestras generadas por el modelo AR (autoregresivo puro):")
print("  El AR genera token por token, condicionado a los anteriores.")
print("  La energía EBM mide qué tan natural es la secuencia completa.")
for i in range(5):
    energy = ar_energies[i].item()
    print(f"  Sample {i}: {ar_samples[i].tolist()} | Energy: {energy:.4f}")
    if energy < 0:
        print("    -> Energia negativa: secuencia de alta calidad, el EBM la acepta.")
    elif energy < 1:
        print("    -> Energia moderada: secuencia plausible pero no perfecta.")
    else:
        print("    -> Energia alta: secuencia poco natural, el EBM la rechaza.")

# 2. Refinar con Langevin en espacio de embeddings
print("\nRefinando con Langevin en embedding space...")
print("  Langevin: movemos embeddings hacia menor energia + ruido para explorar.")
embedded = ar_ebm.ar_model.embedding(ar_samples).clone().requires_grad_(True)
flat = embedded.reshape(5, -1)

step_size = 0.05
for step in range(20):
    energy = ar_ebm.ebm_energy.energy(flat).sum()
    grad = torch.autograd.grad(energy, embedded)[0]
    embedded = embedded - step_size * grad + 0.01 * torch.randn_like(embedded)
    embedded = embedded.detach().requires_grad_(True)
    flat = embedded.reshape(5, -1)

# Decodificar embeddings refinados de vuelta a tokens
# (usando el embedding matrix como projection → nearest neighbor)
with torch.no_grad():
    emb_matrix = ar_ebm.ar_model.embedding.weight  # [vocab, dim]
    refined_flat = embedded.detach().reshape(5 * seq_len, -1)
    sims = refined_flat @ emb_matrix.T  # [5*seq_len, vocab]
    refined_tokens = sims.argmax(dim=-1).reshape(5, seq_len)
    refined_energies = ar_ebm.energy_of_sequence(refined_tokens)

print("\nMuestras refinadas por EBM (despues de Langevin):")
for i in range(5):
    energy = refined_energies[i].item()
    print(f"  Sample {i}: {refined_tokens[i].tolist()} | Energy: {energy:.4f}")
    if energy < 0:
        print("    -> Excelente: el EBM acepta esta secuencia como muy plausible.")
    elif energy < 1:
        print("    -> Buena: secuencia razonable segun el EBM.")
    else:
        print("    -> Mejorable: el EBM aun encuentra esta secuencia poco natural.")

# Comparar energías
avg_ar_energy = ar_energies.mean().item()
avg_refined_energy = refined_energies.mean().item()
improvement = avg_ar_energy - avg_refined_energy
print(f"\nAvg energy AR: {avg_ar_energy:.4f} -> Refined: {avg_refined_energy:.4f}")
if improvement > 0:
    print(f"  -> EBM mejoro las muestras en {improvement:.4f} unidades.")
    print("  -> Langevin funciona: secuencias refinadas son mas reales.")
elif improvement > -0.5:
    print(f"  -> Cambio minimo ({improvement:+.4f}): el AR ya generaba buena calidad.")
else:
    print(f"  -> Energia aumento ({improvement:+.4f}): Langevin muy agresivo.")
    print("  -> Probar step_size mas pequeno o menos pasos.")


---
# Part 4: Synthesis — Conectando los Tres Pilares

## 4.1 El hilo conductor

Todos estos métodos comparten una idea central: **aprender representaciones útiles sin labels**, pero cada uno lo aborda desde un ángulo diferente:

```
                    ¿Cómo evitar el colapso?
                           │
              ┌────────────┼────────────┐
              │            │            │
        Negativos     Regularización   Predicción
        explícitos    en dimensiones   en espacio
        (InfoNCE)     (VICReg, BT)     latente (JEPA)
              │            │            │
              └────────────┼────────────┘
                           │
                    Representaciones
                    compartidas (JEA)
                           │
                    ┌──────┴──────┐
                    │             │
              Discriminativo  Generativo (EBM)
              (similitud)     (energía)
```

## 4.2 Tabla resumen

| Método | Anti-colapso | Batch size | Complejidad | Muestreo |
|--------|-------------|------------|-------------|----------|
| InfoNCE (SimCLR) | Negativos explícitos | Grande (4096+) | O(N²) | No |
| Barlow Twins | Correlación → identidad | Mediano (256+) | O(D²) | No |
| VICReg | Varianza + covarianza | Pequeño (64+) | O(D²) | No |
| DINO/DINOv2 | Teacher-student EMA | Mediano | O(N·D) | No |
| I-JEPA | Predicción latente | Mediano | O(N·D) | No |
| EBM | Función de partición | Cualquiera | O(N) + MCMC | Sí (Langevin) |
| AR + EBM | AR tractable + EBM refinamiento | Cualquiera | O(seq·vocab) + MCMC | Sí (AR + Langevin) |

## 4.3 ¿Cuándo usar cuál?

- **SimCLR/InfoNCE:** Cuando tienes GPU grande y quieres el mejor rendimiento. Los negativos dan señal fuerte.
- **VICReg/Barlow Twins:** Cuando batch size es limitado. Más estable, menos sensible a hiperparámetros.
- **DINO/I-JEPA:** Cuando quieres representaciones ricas sin augmentaciones agresivas. I-JEPA evita reconstruir píxeles.
- **EBMs:** Cuando necesitas modelar la distribución completa (no solo discriminar). Útil para generación.
- **AR + EBM:** Cuando necesitas generación secuencial con calidad global. El AR da estructura, el EBM da coherencia.

In [ ]:
# ============================================================
# Summary: todas las loss functions implementadas
# ============================================================

print("=" * 60)
print("RESUMEN: Loss functions implementadas en este notebook")
print("=" * 60)
print()
print("1. contrastive_loss_ntxent(z_i, z_j, temperature)")
print("   -> InfoNCE / NT-Xent (SimCLR)")
print("   -> Usa negativos intra-batch, temperatura tau")
print("   -> Loss baja (<2): modelo identifica bien el positivo")
print("   -> Loss alta (>4): necesita batch mas grande o tau diferente")
print()
print("2. barlow_twins_loss(z_i, z_j, lambda_coeff)")
print("   -> Barlow Twins (Zbontar et al., 2021)")
print("   -> Matriz de correlacion cruzada -> identidad")
print("   -> Loss baja (<10 para dim=128): representaciones no redundantes")
print("   -> NO puede colapsar: la off-diagonal lo impide")
print()
print("3. vicreg_loss(z_i, z_j, sim_coef, std_coef, cov_coef)")
print("   -> VICReg (Bardes et al., 2022)")
print("   -> Invariance + Variance + Covariance")
print("   -> Loss baja (<30): buen equilibrio entre los 3 terminos")
print("   -> NO puede colapsar: la varianza minima esta forzada")
print()
print("4. ijepa_loss(z_pred, z_target)")
print("   -> I-JEPA (LeCun et al., 2022)")
print("   -> MSE en espacio latente + varianza")
print("   -> Loss baja (<1): el predictor predice bien el target oculto")
print("   -> Evita reconstruir pixeles -- predice en espacio abstracto")
print()
print("5. two_tower_loss(z_a, z_b, temperature)")
print("   -> Two-Tower InfoNCE bidireccional")
print("   -> Cross-modal joint embedding")
print("   -> Accuracy >90%: excelente alineacion cross-modal")
print("   -> La temperatura se aprende -- el modelo ajusta su confianza")
print()
print("6. contrastive_divergence_loss(ebm, x_real, x_fake)")
print("   -> EBM training con MCMC")
print("   -> E(x_real) - E(x_fake)")
print("   -> Loss negativa: el EBM distingue reales de fakes")
print("   -> Loss positiva: necesita mas entrenamiento o mejor MCMC")
print()
print("Arquitecturas implementadas:")
print("  - SimpleEncoder (MLP) -- encoder basico para demos")
print("  - IJEPA -- context encoder + target encoder (EMA) + predictor")
print("  - TwoTowerModel -- cross-modal con temperatura aprendible")
print("  - SimpleEBM -- MLP energy + Langevin sampling")
print("  - AutoregressiveModel -- Transformer decoder causal")
print("  - AutoregressiveEBM -- AR genera + EBM refina con Langevin")
print()
print("=" * 60)
print()
print("Proximos pasos sugeridos:")
print("  1. Probar con datos reales (MNIST, CIFAR-10) en vez de sinteticos")
print("  2. Agregar projector no-lineal (2 capas MLP) antes de la loss")
print("  3. Comparar con DINOv2 pre-entrenado como baseline")
print("  4. Implementar k-NN evaluation para medir calidad de embeddings")
print("  5. Fine-tuning en downstream task (clasificacion) para validar")
print("=" * 60)
